# Stage 3: Real-Time Streaming Pipeline
## Sportlytics Athletics — Professional Basketball Analytics
### Player Telemetry and Alerts 

Consumes live player tracking data from Kafka, computes running per-player metrics (heart rate, cumulative distance), applies fatigue performance thresholds, and generates real-time alerts using Spark Structured Streaming. Writes alert outputs to HDFS and/or console for downstream monitoring.

**Business question:**
During live games, can the system detect when a player's heart rate or cumulative distance exceeds personalized fatigue thresholds and alert the coaching staff within seconds?

## SparkSession with Kafka Support
Import all required libraries and initialize the SparkSession connected to the Spark cluster and HDFS — consistent with Stage 1 and Stage 2.

In [1]:
# Starting the SparkSession
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, from_unixtime, to_timestamp,
    window, count, avg, max as spark_max,
    sum as spark_sum, round as spark_round
)
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

spark = (
    SparkSession.builder
    .appName("Sportlytics-Stage3-Streaming")
    .master("spark://spark-master:7077")
    .config("spark.executor.memory", "1g")
    .config("spark.driver.memory", "1g")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("SparkSession ready!")

Spark version: 3.5.0
SparkSession ready!


## Fatigue Thresholds from Stage 2 batch outputs stored in HDFS
The baselines are computed from historical player tracking data and represent each player’s normal workload levels. They are used to create personalized thresholds for real-time monitoring.
We read `rolling_workload.parquet` written by the Stage 2 batch pipeline and compute per-player thresholds:

- **Heart rate threshold:** 95% of each player's historical maximum heart rate
- **Distance threshold:** each player's historical average per-game distance × 1.1 (10% above their established basel - per the 10% "Red Zone" (Acute:Chronic Workload)i

Players with no historical record fall back:
- Heart rate fallback: 209 BPM (220 × 0.95)
- Distance fallback: 14,520 ft (2.5 miles × 1.1) — based on NBA 2024-25 average of 2.4–2.9 miles per gamene)

In [2]:
# HDFS paths for analytics outputs and streaming checkpoints
ANALYTICS = "hdfs://namenode:9000/sportlytics/analytics"
CHECKPOINT_BASE = "hdfs://namenode:9000/sportlytics/checkpoints/streaming"

In [3]:
# Loading rolling workload Parquet written by Stage 2 batch pipeline
rolling_workload = spark.read.parquet(f"{ANALYTICS}/rolling_workload.parquet")
print("Rolling workload loaded!")

Rolling workload loaded!


In [4]:
# Computing alert thresholds per player based on historical performance
player_thresholds = rolling_workload.groupBy("player_id").agg(
    spark_round(spark_max("avg_heart_rate") * 0.95, 1).alias("hr_alert_threshold"),
    spark_round(avg("total_distance_ft") * 1.10, 1).alias("distance_alert_threshold")
)


print(f"Personalized thresholds derived for {player_thresholds.count():,} players")
player_thresholds.show(10)

Personalized thresholds derived for 450 players
+---------+------------------+------------------------+
|player_id|hr_alert_threshold|distance_alert_threshold|
+---------+------------------+------------------------+
| PLR-0347|             186.2|                   460.9|
| PLR-0388|             209.0|                   448.4|
| PLR-0373|             196.6|                   489.6|
| PLR-0443|             202.4|                   461.6|
| PLR-0382|             190.0|                   436.0|
| PLR-0350|             209.0|                   444.6|
| PLR-0394|             204.3|                   475.9|
| PLR-0390|             207.1|                   459.7|
| PLR-0383|             195.7|                   492.9|
| PLR-0370|             204.3|                   449.0|
+---------+------------------+------------------------+
only showing top 10 rows



## Kafka Message Schema
The schema matches the fields produced by `sportlytics_event_producer.py` and covers the fields specified in the project requirements:
**game_id, player_id, timestamp, speed, heart_rate, cumulative_distance**

In [5]:
# Defining Spark schema for Kafka telemetry data 
telemetry_schema = StructType([
    StructField("game_id",            StringType(), True),
    StructField("player_id",          StringType(), True),
    StructField("team_id",            StringType(), True),
    StructField("game_timestamp",     StringType(), True),
    StructField("event_time",         DoubleType(), True),  
    StructField("x_court_ft",         DoubleType(), True),
    StructField("y_court_ft",         DoubleType(), True),
    StructField("speed_mph",          DoubleType(), True),
    StructField("acceleration",       DoubleType(), True),
    StructField("cumulative_distance",DoubleType(), True),  
    StructField("heart_rate_bpm",     DoubleType(), True),
])

print("Schema defined!")

Schema defined!


## Kafka Stream Data Parsing
Spark Structured Streaming reads from the `realtime_player_telemetry` Kafka topic as an unbounded DataFrame. Each message value arrives as raw bytes and is cast to a JSON string before being parsed into typed columns using the schema defined above.

In [6]:
# Reading live player telemetry stream from Kafka and parse JSON into structured columns
gps_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "realtime_player_telemetry")
    .option("startingOffsets", "earliest")
    .load()
)

telemetry_parsed = (
    gps_raw
    .selectExpr("CAST(value AS STRING) as json_str")
    .select(from_json(col("json_str"), telemetry_schema).alias("data"))
    .select("data.*")
    .withColumn("event_timestamp", to_timestamp(from_unixtime(col("event_time"))))
    # Apply same heart rate validation as Stage 2 cleaning (60-220 BPM)
    .filter((col("heart_rate_bpm") >= 60) & (col("heart_rate_bpm") <= 220))
)

print("Streaming DataFrame ready!")
telemetry_parsed.printSchema()

Streaming DataFrame ready!
root
 |-- game_id: string (nullable = true)
 |-- player_id: string (nullable = true)
 |-- team_id: string (nullable = true)
 |-- game_timestamp: string (nullable = true)
 |-- event_time: double (nullable = true)
 |-- x_court_ft: double (nullable = true)
 |-- y_court_ft: double (nullable = true)
 |-- speed_mph: double (nullable = true)
 |-- acceleration: double (nullable = true)
 |-- cumulative_distance: double (nullable = true)
 |-- heart_rate_bpm: double (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)



## Player Activity Tracking with Tumbling Window
We are grouping data by **player_id** into **60 second tumbling window** to calculate each player’s total exertion, and use withWatermark("event_timestamp", "10 seconds") to handle late data by clearing old state and preventing memory issues.

In [7]:
# Creating 60 second windowed exertion metrics per player, team, and game
# Watermark Manages late events and save memory
windowed_exertion = (
    telemetry_parsed
    .withWatermark("event_timestamp", "10 seconds")
    .groupBy(
        window(col("event_timestamp"), "60 seconds"),
        col("player_id"),
        col("team_id"),
        col("game_id")
    )
    .agg(
        spark_round(avg("heart_rate_bpm"), 1).alias("avg_heart_rate"),
        spark_round(spark_max("heart_rate_bpm"), 1).alias("max_heart_rate"),
        spark_round(spark_sum("cumulative_distance"), 1).alias("total_distance_ft"),
        spark_round(avg("speed_mph"), 2).alias("avg_speed_mph"),
        count("*").alias("tracking_frames")
    )
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("player_id"),
        col("team_id"),
        col("game_id"),
        col("avg_heart_rate"),
        col("max_heart_rate"),
        col("total_distance_ft"),
        col("avg_speed_mph"),
        col("tracking_frames")
    )
)

print("Windowed aggregation defined!")

Windowed aggregation defined!


## Fatigue Thresholds
Two alerts are triggered using personalized thresholds derived from Stage 2:
- **Heart rate alert:** average heart rate in the window >= 95% of the player's historical max
- **Distance alert:** cumulative distance in the window >= player's historical average × 1.1

Thresholds are joined from the Stage 2 batch output as a static DataFrame. 

In [8]:
# Building player fatigue alerts by applying individual thresholds for heart rate and distance.
alerts_df = (
    windowed_exertion
    .join(player_thresholds, on="player_id", how="left")
    # Fill nulls for players not in historical data with population defaults
    .fillna({
        "hr_alert_threshold": 209.0,
        "distance_alert_threshold": 14520.0
    })
    # Alert 1: heart rate exceeds 95% of player's known max
    .withColumn("heart_rate_alert", col("avg_heart_rate") >= col("hr_alert_threshold"))
    # Alert 2: cumulative distance exceeds fatigue threshold
    .withColumn("distance_alert", col("total_distance_ft") >= col("distance_alert_threshold"))
    # Combined flag
    .withColumn("fatigue_alert", col("heart_rate_alert") | col("distance_alert"))
)

print("Alert thresholds applied!")
alerts_df.printSchema()

Alert thresholds applied!
root
 |-- player_id: string (nullable = true)
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)
 |-- team_id: string (nullable = true)
 |-- game_id: string (nullable = true)
 |-- avg_heart_rate: double (nullable = true)
 |-- max_heart_rate: double (nullable = true)
 |-- total_distance_ft: double (nullable = true)
 |-- avg_speed_mph: double (nullable = true)
 |-- tracking_frames: long (nullable = false)
 |-- hr_alert_threshold: double (nullable = false)
 |-- distance_alert_threshold: double (nullable = false)
 |-- heart_rate_alert: boolean (nullable = true)
 |-- distance_alert: boolean (nullable = true)
 |-- fatigue_alert: boolean (nullable = true)



## Memory Sink Storage and Live Results
The streaming query writes the windowed results to an in-memory table called player_exertion, so you can query it in real time using spark.sql().

Since this is a windowed aggregation, the values keep updating as new data comes in. Using complete mode makes sure the full results are written every time, so coaches always see the most up-to-date totals.

The checkpoint in HDFS makes sure Spark picks up from the last Kafka offset if it restarts, so nothing gets reprocessed or duplicated.

In [9]:
import time

# Start streaming query writing to memory sink for live querying
exertion_query = (
    alerts_df
    .writeStream
    .outputMode("complete")
    .format("memory")
    .queryName("player_exertion")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/player_exertion")
    .start()
)

print("Streaming query started — waiting 30 seconds for micro-batches...")
time.sleep(30)
print(f"Query status: {exertion_query.status}")

Streaming query started — waiting 30 seconds for micro-batches...
Query status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}


In [30]:
print(spark.streams.active)
print(exertion_query.status)

{'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}


## Live Query Results and Findings
The queries below identify which players are at elevated fatigue risk during a live game. Results are ordered by alert status and heart rate so coaching staff see the highest-risk players first.

In [10]:
# Each player exertion metrics across all windows
print(f"Total windows processed: {spark.sql('SELECT COUNT(*) FROM player_exertion').collect()[0][0]:,}")
spark.sql("""
    SELECT
        player_id,
        team_id,
        game_id,
        window_start,
        window_end,
        avg_heart_rate,
        max_heart_rate,
        hr_alert_threshold,
        ROUND(total_distance_ft / 5280, 2) AS distance_miles,
        ROUND(distance_alert_threshold / 5280, 2) AS threshold_miles,
        tracking_frames,
        heart_rate_alert,
        distance_alert,
        fatigue_alert
    FROM player_exertion
    ORDER BY fatigue_alert DESC, avg_heart_rate DESC
""").show(20, truncate=False)

Total windows processed: 296,177
+---------+-------+----------+-------------------+-------------------+--------------+--------------+------------------+--------------+---------------+---------------+----------------+--------------+-------------+
|player_id|team_id|game_id   |window_start       |window_end         |avg_heart_rate|max_heart_rate|hr_alert_threshold|distance_miles|threshold_miles|tracking_frames|heart_rate_alert|distance_alert|fatigue_alert|
+---------+-------+----------+-------------------+-------------------+--------------+--------------+------------------+--------------+---------------+---------------+----------------+--------------+-------------+
|PLR-0101 |TEAM-26|GAME-00144|2026-05-03 05:07:00|2026-05-03 05:08:00|220.0         |220.0         |205.2             |0.02          |0.09           |1              |true            |false         |true         |
|PLR-0175 |TEAM-18|GAME-00651|2026-05-03 04:18:00|2026-05-03 04:19:00|220.0         |220.0         |209.0          

In [11]:
# Heart rate alerts — players whose avg heart rate exceeds 95% of their historical max
spark.sql("""
    SELECT
        player_id,
        team_id,
        game_id,
        avg_heart_rate,
        hr_alert_threshold,
        heart_rate_alert
    FROM player_exertion
    WHERE heart_rate_alert = TRUE
    ORDER BY avg_heart_rate DESC
""").show(20, truncate=False)

+---------+-------+----------+--------------+------------------+----------------+
|player_id|team_id|game_id   |avg_heart_rate|hr_alert_threshold|heart_rate_alert|
+---------+-------+----------+--------------+------------------+----------------+
|PLR-0265 |TEAM-04|GAME-00014|220.0         |203.3             |true            |
|PLR-0296 |TEAM-13|GAME-00450|220.0         |193.8             |true            |
|PLR-0070 |TEAM-24|GAME-00517|220.0         |209.0             |true            |
|PLR-0159 |TEAM-15|GAME-01685|220.0         |191.9             |true            |
|PLR-0221 |TEAM-20|GAME-00717|220.0         |194.8             |true            |
|PLR-0054 |TEAM-16|GAME-00535|220.0         |206.1             |true            |
|PLR-0269 |TEAM-30|GAME-01167|220.0         |209.0             |true            |
|PLR-0265 |TEAM-04|GAME-00694|220.0         |203.3             |true            |
|PLR-0438 |TEAM-22|GAME-00689|220.0         |198.5             |true            |
|PLR-0247 |TEAM-

In [12]:
# Distance alerts — players whose cumulative distance exceeds their fatigue threshold
spark.sql("""
    SELECT
        player_id,
        team_id,
        game_id,
        ROUND(total_distance_ft / 5280, 2) AS distance_miles,
        ROUND(distance_alert_threshold / 5280, 2) AS threshold_miles,
        distance_alert
    FROM player_exertion
    WHERE distance_alert = TRUE
    ORDER BY total_distance_ft DESC
""").show(20, truncate=False)

+---------+-------+----------+--------------+---------------+--------------+
|player_id|team_id|game_id   |distance_miles|threshold_miles|distance_alert|
+---------+-------+----------+--------------+---------------+--------------+
|PLR-0445 |TEAM-08|GAME-01113|0.18          |0.09           |true          |
|PLR-0356 |TEAM-24|GAME-01822|0.18          |0.09           |true          |
|PLR-0326 |TEAM-13|GAME-00510|0.17          |0.09           |true          |
|PLR-0068 |TEAM-17|GAME-00042|0.17          |0.1            |true          |
|PLR-0313 |TEAM-06|GAME-00281|0.17          |0.09           |true          |
|PLR-0445 |TEAM-08|GAME-01208|0.17          |0.09           |true          |
|PLR-0125 |TEAM-23|GAME-00673|0.17          |0.09           |true          |
|PLR-0297 |TEAM-22|GAME-01995|0.16          |0.08           |true          |
|PLR-0260 |TEAM-01|GAME-00936|0.16          |0.09           |true          |
|PLR-0162 |TEAM-19|GAME-01850|0.16          |0.09           |true          |

In [13]:
# Players currently flagged — coaching staff substitution candidates
# These are players whose heart rate or distance has exceeded their personal threshold
# giving coaches data-driven evidence for substitution decisions in real time
print(f"Total fatigue alerts: {spark.sql('SELECT COUNT(*) FROM player_exertion WHERE fatigue_alert = TRUE').collect()[0][0]:,}")
spark.sql("""
    SELECT
        player_id,
        team_id,
        game_id,
        avg_heart_rate,
        hr_alert_threshold,
        ROUND(total_distance_ft / 5280, 2) AS distance_miles,
        ROUND(distance_alert_threshold / 5280, 2) AS threshold_miles,
        heart_rate_alert,
        distance_alert
    FROM player_exertion
    WHERE fatigue_alert = TRUE
    ORDER BY avg_heart_rate DESC
""").show(20, truncate=False)

Total fatigue alerts: 19,937
+---------+-------+----------+--------------+------------------+--------------+---------------+----------------+--------------+
|player_id|team_id|game_id   |avg_heart_rate|hr_alert_threshold|distance_miles|threshold_miles|heart_rate_alert|distance_alert|
+---------+-------+----------+--------------+------------------+--------------+---------------+----------------+--------------+
|PLR-0265 |TEAM-04|GAME-00014|220.0         |203.3             |0.02          |0.08           |true            |false         |
|PLR-0168 |TEAM-04|GAME-00864|220.0         |194.8             |0.06          |0.1            |true            |false         |
|PLR-0070 |TEAM-24|GAME-00517|220.0         |209.0             |0.07          |0.09           |true            |false         |
|PLR-0237 |TEAM-16|GAME-01325|220.0         |207.1             |0.07          |0.08           |true            |false         |
|PLR-0175 |TEAM-18|GAME-00651|220.0         |209.0             |0.01       

In [14]:
# Alert summary by team — identifies which teams have the most fatigued players
# Teams with high alert counts may need to adjust rotation strategy or rest key players
# This directly supports the league's collective bargaining requirement to track
# and report player workload data as a compliance tool
print(f"Alert summary by team:")
spark.sql("""
    SELECT
        team_id,
        COUNT(DISTINCT player_id)                                AS players_monitored,
        SUM(CASE WHEN fatigue_alert    = TRUE THEN 1 ELSE 0 END) AS total_alerts,
        SUM(CASE WHEN heart_rate_alert = TRUE THEN 1 ELSE 0 END) AS hr_alerts,
        SUM(CASE WHEN distance_alert   = TRUE THEN 1 ELSE 0 END) AS distance_alerts,
        ROUND(AVG(avg_heart_rate), 1)                            AS team_avg_hr
    FROM player_exertion
    GROUP BY team_id
    ORDER BY total_alerts DESC
""").show(30, truncate=False)

Alert summary by team:
+-------+-----------------+------------+---------+---------------+-----------+
|team_id|players_monitored|total_alerts|hr_alerts|distance_alerts|team_avg_hr|
+-------+-----------------+------------+---------+---------------+-----------+
|TEAM-10|15               |866         |115      |756            |144.5      |
|TEAM-06|15               |808         |149      |662            |144.3      |
|TEAM-16|15               |794         |95       |704            |144.3      |
|TEAM-07|15               |792         |223      |587            |144.9      |
|TEAM-20|15               |749         |144      |608            |144.8      |
|TEAM-29|15               |745         |137      |624            |144.3      |
|TEAM-03|15               |738         |165      |578            |144.4      |
|TEAM-19|15               |726         |134      |599            |145.4      |
|TEAM-27|15               |724         |170      |563            |144.8      |
|TEAM-22|15               |68

## Streaming Query Health
Spark Structured Streaming exposes query progress metrics.This verifies the pipeline is actually running and processing events and the Stage 4 Airflow DAG uses these metrics to monitor streaming health.

In [15]:
print("=== Streaming Query Status ===")
for query in spark.streams.active:
    print(f"\nQuery       : {query.name}")
    print(f"Is active   : {query.isActive}")
    if query.lastProgress:
        prog = query.lastProgress
        print(f"Batch ID    : {prog.get('batchId', 'N/A')}")
        print(f"Input rows  : {prog.get('numInputRows', 0):,}")
        print(f"Input rate  : {prog.get('inputRowsPerSecond', 0):.2f} rows/sec")
        print(f"Process rate: {prog.get('processedRowsPerSecond', 0):.2f} rows/sec")

=== Streaming Query Status ===

Query       : player_exertion
Is active   : True
Batch ID    : 687
Input rows  : 84
Input rate  : 24.55 rows/sec
Process rate: 28.10 rows/sec


## Fatigue Alerts to HDFS
Persists only the flagged alert records to HDFS so the Stage 4 Airflow DAG can pick them up for downstream processing and reporting.
Uses `append` mode so only newly triggered alerts are written each micro-batch.

In [16]:
ALERTS_PATH = "hdfs://namenode:9000/sportlytics/analytics/streaming_alerts.parquet"

alert_sink_query = (
    alerts_df
    .filter(col("fatigue_alert") == True)
    .writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", ALERTS_PATH)
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/fatigue_alerts")
    .start()
)

print(f"Writing fatigue alerts to: {ALERTS_PATH}")
time.sleep(30)

try:
    written = spark.read.parquet(ALERTS_PATH)
    print(f"Alerts written to HDFS: {written.count():,} records")
    written.orderBy("avg_heart_rate", ascending=False).show(5, truncate=False)
except Exception as e:
    print(f"No alerts written yet: {e}")

Writing fatigue alerts to: hdfs://namenode:9000/sportlytics/analytics/streaming_alerts.parquet
Alerts written to HDFS: 20,179 records
+---------+-------------------+-------------------+-------+----------+--------------+--------------+-----------------+-------------+---------------+------------------+------------------------+----------------+--------------+-------------+
|player_id|window_start       |window_end         |team_id|game_id   |avg_heart_rate|max_heart_rate|total_distance_ft|avg_speed_mph|tracking_frames|hr_alert_threshold|distance_alert_threshold|heart_rate_alert|distance_alert|fatigue_alert|
+---------+-------------------+-------------------+-------+----------+--------------+--------------+-----------------+-------------+---------------+------------------+------------------------+----------------+--------------+-------------+
|PLR-0093 |2026-05-03 04:44:00|2026-05-03 04:45:00|TEAM-27|GAME-01208|220.0         |220.0         |470.6            |5.5          |1              |2

## Optional: Streaming Query Termination Control
The cell below can be run when the demo is complete to cleanly stop all active streaming queries. Checkpoints will be preserved in HDFS so the pipeline can be resumed from the last committed Kafka offset.

#### Stopping all active streaming queries cleanly
for query in spark.streams.active:
    print(f"Stopping: {query.name}")
    query.stop()

print("All streaming queries stopped.")

print(f"Checkpoints saved at: {CHECKPOINT_BASE}")

print("Re-run the memory sink and HDFS sink cells to resume from last committed Kafka offset.")